# Data Preparation for Evaluation Set

This notebook performs data preprocessing and feature engineering for the evaluation (eval) campaign donation dataset.

## Tasks:
1. **Data Loading**: Load donor data from eval set JSON files
2. **Preprocessing**: Clean and convert data types (amount, campaign_duration, target_dana)
3. **Feature Engineering**: Calculate progress ratio and create donation amount bins
4. **Probability Distribution**: Calculate probability distribution of donors across bins per campaign
5. **Output**: Create final dataset with 1 row per campaign containing all features

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import glob
import os
import re
import json
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Data Loading

Load JSON files from the eval dataset folder.

In [2]:
# Define paths
data_folder = Path('../dataset/eval')
output_folder = Path('.')

print(f"Loading eval data from: {data_folder.absolute()}")

# Find all JSON files in the eval folder
json_files = list(data_folder.glob('donors_*.json'))

if len(json_files) == 0:
    raise FileNotFoundError(f"No JSON files found in {data_folder}")
else:
    print(f"\n✓ Found {len(json_files)} eval donor JSON files")

print("\nFiles:")
for f in json_files:
    print(f"  - {f.name}")

Loading eval data from: /Users/enjat/Github/data-mining-t2-2025/data-preparation/../dataset/eval

✓ Found 5 eval donor JSON files

Files:
  - donors_20251221_203249.json
  - donors_20251221_203213.json
  - donors_20251221_203035.json
  - donors_20251221_202957.json
  - donors_20251221_202932.json


### 1.1 Helper Functions

In [3]:
def load_json_file(json_file):
    """
    Load JSON file and extract donor data with metadata.
    """
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Extract metadata
    metadata = data.get('campaign_metadata', {})
    if not metadata:
        metadata = data.get('data', {})

    N = metadata.get('N', '')
    campaign_duration = metadata.get('campaign_duration', '')
    target_dana = metadata.get('target_dana', '')

    # Extract donors
    donors = data.get('donors', [])

    # Convert to DataFrame
    if len(donors) > 0:
        df = pd.DataFrame(donors)
        # Add metadata columns to every row
        df['N'] = N
        df['campaign_duration'] = campaign_duration
        df['target_dana'] = target_dana
        return df
    else:
        return pd.DataFrame(columns=['name', 'amount', 'time_ago', 'N', 'campaign_duration', 'target_dana'])

### 1.2 Load and Process Files

In [4]:
all_dataframes = []

print(f"{'='*60}")
print("Processing Eval Files...")
print(f"{'='*60}")

for json_file in json_files:
    try:
        # Load JSON file
        df = load_json_file(json_file)

        # Use filename as unique identifier
        file_identifier = json_file.name
        df['campaign'] = file_identifier

        all_dataframes.append(df)
        print(f"Loaded {json_file.name}: {len(df)} rows")

    except Exception as e:
        print(f"⚠ Error loading {json_file.name}: {e}")
        continue

# Merge all dataframes
if len(all_dataframes) > 0:
    df_donors = pd.concat(all_dataframes, ignore_index=True)
    print(f"\n✓ Successfully merged {len(df_donors):,} donor records.")
else:
    raise ValueError("No data loaded from JSON files.")

print(f"\n{'='*60}")
print(f"Data Summary:")
print(f"{'='*60}")
print(f"Total merged rows: {len(df_donors):,}")
print(f"Total unique campaigns: {df_donors['campaign'].nunique()}")
print(f"Columns: {list(df_donors.columns)}")
print(f"{'='*60}")

# Display sample data
print(f"\nSample data:")
print(df_donors.head())

Processing Eval Files...
Loaded donors_20251221_203249.json: 253 rows
Loaded donors_20251221_203213.json: 1675 rows
Loaded donors_20251221_203035.json: 416 rows
Loaded donors_20251221_202957.json: 50 rows
Loaded donors_20251221_202932.json: 36 rows

✓ Successfully merged 2,430 donor records.

Data Summary:
Total merged rows: 2,430
Total unique campaigns: 5
Columns: ['name', 'amount', 'time_ago', 'N', 'campaign_duration', 'target_dana', 'campaign']

Sample data:
              name    amount           time_ago    N  \
0  Masjid AT TAQWA  Rp50.000  sebulan yang lalu  253   
1      Mrs Diamond   Rp2.000  sebulan yang lalu  253   
2       Orang Baik  Rp20.000  sebulan yang lalu  253   
3      Mrs Diamond   Rp2.000  sebulan yang lalu  253   
4      Mrs Diamond   Rp2.000  sebulan yang lalu  253   

            campaign_duration    target_dana                     campaign  
0  4411 (days remaining only)  Rp300.000.000  donors_20251221_203249.json  
1  4411 (days remaining only)  Rp300.000.000 

## 2. Data Preprocessing

### 2.1 Convert Amount to Numeric

In [5]:
def convert_rupiah_to_number(value):
    """
    Convert Rupiah format string to numeric value.
    Example: 'Rp5.000' -> 5000.0, 'Rp250.000.000' -> 250000000.0
    """
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)

    # Remove 'Rp', dots, commas, and whitespace
    cleaned = str(value).replace('Rp', '').replace('.', '').replace(',', '').strip()

    try:
        return float(cleaned)
    except (ValueError, AttributeError):
        return np.nan

# Convert amount column to numeric
df_donors['amount_numeric'] = df_donors['amount'].apply(convert_rupiah_to_number)

print(f"Amount conversion summary:")
print(f"  - Total rows: {len(df_donors):,}")
print(f"  - Valid amounts: {df_donors['amount_numeric'].notna().sum():,}")
print(f"  - Invalid amounts: {df_donors['amount_numeric'].isna().sum():,}")
print(f"\nSample conversion:")
print(df_donors[['amount', 'amount_numeric']].head(10))

Amount conversion summary:
  - Total rows: 2,430
  - Valid amounts: 2,430
  - Invalid amounts: 0

Sample conversion:
     amount  amount_numeric
0  Rp50.000        50000.00
1   Rp2.000         2000.00
2  Rp20.000        20000.00
3   Rp2.000         2000.00
4   Rp2.000         2000.00
5   Rp2.000         2000.00
6   Rp5.000         5000.00
7   Rp5.000         5000.00
8   Rp7.000         7000.00
9   Rp5.000         5000.00


### 2.2 Extract Campaign Duration (Integer Days)

In [6]:
def extract_duration_days(duration_str):
    """
    Extract integer days from campaign_duration string.
    """
    if pd.isna(duration_str):
        return 0

    duration_str = str(duration_str).strip()
    match = re.search(r'(\d+)', duration_str)

    if match:
        return int(match.group(1))
    else:
        return 0

df_donors['campaign_duration_days'] = df_donors['campaign_duration'].apply(extract_duration_days)

print(f"Campaign duration extraction summary:")
print(f"\nUnique duration values:")
print(df_donors['campaign_duration'].value_counts())
print(f"\nExtracted days (sample):")
print(df_donors[['campaign_duration', 'campaign_duration_days']].drop_duplicates())

Campaign duration extraction summary:

Unique duration values:
campaign_duration
740 (days remaining only)     1675
598 (days remaining only)      416
4411 (days remaining only)     253
10 (days remaining only)        50
24 (days remaining only)        36
Name: count, dtype: int64

Extracted days (sample):
               campaign_duration  campaign_duration_days
0     4411 (days remaining only)                    4411
253    740 (days remaining only)                     740
1928   598 (days remaining only)                     598
2344    10 (days remaining only)                      10
2394    24 (days remaining only)                      24


### 2.3 Convert Target Dana to Numeric

In [7]:
df_donors['target_dana_numeric'] = df_donors['target_dana'].apply(convert_rupiah_to_number)

print(f"Target dana conversion summary:")
print(f"  - Valid targets: {df_donors['target_dana_numeric'].notna().sum():,}")
print(f"  - Invalid targets: {df_donors['target_dana_numeric'].isna().sum():,}")
print(f"\nSample conversion:")
print(df_donors[['target_dana', 'target_dana_numeric', 'campaign']].drop_duplicates())

Target dana conversion summary:
  - Valid targets: 2,430
  - Invalid targets: 0

Sample conversion:
        target_dana  target_dana_numeric                     campaign
0     Rp300.000.000         300000000.00  donors_20251221_203249.json
253   Rp500.000.000         500000000.00  donors_20251221_203213.json
1928  Rp426.000.000         426000000.00  donors_20251221_203035.json
2344  Rp100.000.000         100000000.00  donors_20251221_202957.json
2394   Rp40.900.000          40900000.00  donors_20251221_202932.json


### 2.4 Clean Data

In [8]:
original_count = len(df_donors)
original_campaigns = df_donors['campaign'].nunique()

# Remove rows with invalid amounts
df_clean = df_donors.dropna(subset=['amount_numeric'])
df_clean = df_clean[df_clean['amount_numeric'] > 0]

print(f"Data cleaning summary:")
print(f"  - Original rows: {original_count:,}")
print(f"  - Original campaigns: {original_campaigns}")
print(f"  - After cleaning: {len(df_clean):,}")
print(f"  - Removed rows: {original_count - len(df_clean):,}")
print(f"  - Remaining campaigns: {df_clean['campaign'].nunique()}")

print(f"\n{'='*60}")
print(f"Dataset Statistics After Cleaning")
print(f"{'='*60}")
print(f"Total donations: {len(df_clean):,}")
print(f"Total amount: Rp {df_clean['amount_numeric'].sum():,.0f}")
print(f"Min donation: Rp {df_clean['amount_numeric'].min():,.0f}")
print(f"Max donation: Rp {df_clean['amount_numeric'].max():,.0f}")
print(f"Average donation: Rp {df_clean['amount_numeric'].mean():,.2f}")
print(f"Median donation: Rp {df_clean['amount_numeric'].median():,.0f}")

Data cleaning summary:
  - Original rows: 2,430
  - Original campaigns: 5
  - After cleaning: 2,430
  - Removed rows: 0
  - Remaining campaigns: 5

Dataset Statistics After Cleaning
Total donations: 2,430
Total amount: Rp 56,069,586
Min donation: Rp 1,000
Max donation: Rp 10,000,000
Average donation: Rp 23,073.90
Median donation: Rp 5,000


## 3. Feature Engineering

### 3.1 Calculate Progress Ratio

In [9]:
# Calculate total amount collected per campaign
campaign_totals = df_clean.groupby('campaign').agg({
    'amount_numeric': 'sum',
    'target_dana_numeric': 'first',
    'campaign_duration_days': 'first'
}).reset_index()

campaign_totals.columns = ['campaign', 'total_collected', 'target_dana', 'campaign_duration']
campaign_totals['campaign_duration'] = campaign_totals['campaign_duration'].fillna(0).astype(int)
campaign_totals['progress'] = campaign_totals['total_collected'] / campaign_totals['target_dana'].replace(0, np.nan)
campaign_totals['progress'] = campaign_totals['progress'].fillna(0)

# Get N (total donors per campaign)
campaign_counts = df_clean.groupby('campaign').size().reset_index(name='N')
campaign_totals = campaign_totals.merge(campaign_counts, on='campaign')

print("Progress calculation summary:")
print(f"\nTotal campaigns: {len(campaign_totals)}")
print(f"\nProgress statistics:")
print(f"  - Min: {campaign_totals['progress'].min():.4f}")
print(f"  - Max: {campaign_totals['progress'].max():.4f}")
print(f"  - Average: {campaign_totals['progress'].mean():.4f}")
print(f"  - Median: {campaign_totals['progress'].median():.4f}")

display_cols = ['campaign', 'N', 'total_collected', 'target_dana', 'progress']
print(f"\nCampaign details:")
print(campaign_totals[display_cols].to_string(index=False))

Progress calculation summary:

Total campaigns: 5

Progress statistics:
  - Min: 0.0072
  - Max: 0.0824
  - Average: 0.0306
  - Median: 0.0162

Campaign details:
                   campaign    N  total_collected  target_dana  progress
donors_20251221_202932.json   36        295000.00  40900000.00      0.01
donors_20251221_202957.json   50       3244000.00 100000000.00      0.03
donors_20251221_203035.json  416       6907000.00 426000000.00      0.02
donors_20251221_203213.json 1675      41212000.00 500000000.00      0.08
donors_20251221_203249.json  253       4411586.00 300000000.00      0.01


## 4. Binning and Probability Distribution

### 4.1 Define Bins (Same as Training)

In [10]:
# Use the same bins as training data
custom_bins = [0, 10000, 25000, 50000, 100000, float('inf')]
bin_labels = [f'p{i+1}' for i in range(len(custom_bins)-1)]

# Apply binning
df_clean['amount_bin'] = pd.cut(
    df_clean['amount_numeric'],
    bins=custom_bins,
    labels=bin_labels,
    include_lowest=True,
    right=True
)

# Calculate statistics per bin
bin_stats = df_clean.groupby('amount_bin', observed=False)['amount_numeric'].agg(['mean', 'median', 'min', 'max', 'count'])

print(f"{'='*60}")
print("Bin Statistics for Eval Set:")
print(f"{'='*60}")

for i, label in enumerate(bin_labels):
    lower = custom_bins[i]
    upper = custom_bins[i+1]
    display_lower = lower if i == 0 else lower + 1
    range_str = f"> {lower:,.0f}" if upper == float('inf') else f"{display_lower:,.0f} - {upper:,.0f}"

    mean_val = bin_stats.loc[label, 'mean']
    median_val = bin_stats.loc[label, 'median']
    min_val = bin_stats.loc[label, 'min']
    max_val = bin_stats.loc[label, 'max']
    count_val = bin_stats.loc[label, 'count']

    if pd.notna(mean_val):
        print(f"  {label} ({range_str}):")
        print(f"     -> Count  : {count_val:,} donors")
        print(f"     -> Min    : Rp {min_val:,.0f}")
        print(f"     -> Max    : Rp {max_val:,.0f}")
        print(f"     -> Median : Rp {median_val:,.0f}")
        print(f"     -> Mean   : Rp {mean_val:,.0f}")
        print("-" * 40)

# Distribution
print(f"\n{'='*60}")
print("Distribution per Bin:")
print(f"{'='*60}")
bin_counts = df_clean['amount_bin'].value_counts().sort_index()
bin_proportions = bin_counts / len(df_clean)

for bin_label in bin_labels:
    count = bin_counts.get(bin_label, 0)
    prop = bin_proportions.get(bin_label, 0)
    print(f"  {bin_label}: {count:6,} donors ({prop*100:5.2f}%)")

Bin Statistics for Eval Set:
  p1 (0 - 10,000):
     -> Count  : 1,806 donors
     -> Min    : Rp 1,000
     -> Max    : Rp 10,000
     -> Median : Rp 2,000
     -> Mean   : Rp 3,699
----------------------------------------
  p2 (10,001 - 25,000):
     -> Count  : 184 donors
     -> Min    : Rp 11,000
     -> Max    : Rp 25,000
     -> Median : Rp 20,000
     -> Mean   : Rp 20,310
----------------------------------------
  p3 (25,001 - 50,000):
     -> Count  : 368 donors
     -> Min    : Rp 27,000
     -> Max    : Rp 50,000
     -> Median : Rp 50,000
     -> Mean   : Rp 49,147
----------------------------------------
  p4 (50,001 - 100,000):
     -> Count  : 52 donors
     -> Min    : Rp 53,000
     -> Max    : Rp 100,000
     -> Median : Rp 100,000
     -> Mean   : Rp 94,481
----------------------------------------
  p5 (> 100,000):
     -> Count  : 20 donors
     -> Min    : Rp 135,000
     -> Max    : Rp 10,000,000
     -> Median : Rp 260,000
     -> Mean   : Rp 1,132,679
---------

### 4.2 Calculate Probability Distribution per Campaign

In [11]:
# Group by campaign and bin
campaign_bin_counts = df_clean.groupby(['campaign', 'amount_bin'], observed=False).size().reset_index(name='count')

# Pivot to get one row per campaign
campaign_bin_pivot = campaign_bin_counts.pivot(index='campaign', columns='amount_bin', values='count').fillna(0)

# Ensure all bin columns exist
for label in bin_labels:
    if label not in campaign_bin_pivot.columns:
        campaign_bin_pivot[label] = 0

campaign_bin_pivot = campaign_bin_pivot[bin_labels]
campaign_bin_pivot['N_calc'] = campaign_bin_pivot[bin_labels].sum(axis=1)

print("Campaign bin counts:")
print(campaign_bin_pivot)

# Calculate probabilities
campaign_probs = campaign_bin_pivot.copy()
for label in bin_labels:
    campaign_probs[label] = campaign_probs[label].div(campaign_probs['N_calc'].replace(0, np.nan)).fillna(0)

campaign_probs = campaign_probs.drop(columns=['N_calc'])

print(f"\nProbability distribution:")
print(campaign_probs)

# Verify probabilities sum to 1
prob_sums = campaign_probs[bin_labels].sum(axis=1)
print(f"\nProbability sum verification:")
print(f"  - Min sum: {prob_sums.min():.6f}")
print(f"  - Max sum: {prob_sums.max():.6f}")
print(f"  - All sums ≈ 1? {np.allclose(prob_sums, 1.0)}")

Campaign bin counts:
amount_bin                     p1  p2   p3  p4  p5  N_calc
campaign                                                  
donors_20251221_202932.json    31   3    2   0   0      36
donors_20251221_202957.json    36   6    4   3   1      50
donors_20251221_203035.json   297  77   28   9   5     416
donors_20251221_203213.json  1246  75  319  25  10    1675
donors_20251221_203249.json   196  23   15  15   4     253

Probability distribution:
amount_bin                    p1   p2   p3   p4   p5
campaign                                            
donors_20251221_202932.json 0.86 0.08 0.06 0.00 0.00
donors_20251221_202957.json 0.72 0.12 0.08 0.06 0.02
donors_20251221_203035.json 0.71 0.19 0.07 0.02 0.01
donors_20251221_203213.json 0.74 0.04 0.19 0.01 0.01
donors_20251221_203249.json 0.77 0.09 0.06 0.06 0.02

Probability sum verification:
  - Min sum: 1.000000
  - Max sum: 1.000000
  - All sums ≈ 1? True


## 5. Final Output Formatting

In [12]:
# Merge features
df_final = campaign_totals[['campaign', 'N', 'campaign_duration', 'target_dana', 'progress']].copy()
df_final = df_final.merge(campaign_probs.reset_index(), on='campaign', how='outer')

# Fill missing values
prob_cols = [col for col in df_final.columns if col.startswith('p')]
df_final[prob_cols] = df_final[prob_cols].fillna(0)
df_final['N'] = df_final['N'].fillna(0).astype(int)
df_final['campaign_duration'] = df_final['campaign_duration'].fillna(0).astype(int)
df_final['target_dana'] = df_final['target_dana'].fillna(0).astype(float)
df_final['progress'] = df_final['progress'].fillna(0).astype(float)

# Reorder columns
column_order = ['campaign', 'N', 'campaign_duration', 'target_dana', 'progress'] + bin_labels
df_final = df_final[column_order]

print("Final eval dataset structure:")
print(f"  - Shape: {df_final.shape}")
print(f"  - Columns: {list(df_final.columns)}")
print(f"\nFinal dataset:")
print(df_final.to_string(index=False))

print(f"\n{'='*60}")
print("Eval Dataset Summary")
print(f"{'='*60}")
print(f"Total campaigns: {len(df_final)}")
print(f"Total donors: {df_final['N'].sum():,}")
print(f"\nCampaign duration statistics:")
print(f"  - Min: {df_final['campaign_duration'].min()} days")
print(f"  - Max: {df_final['campaign_duration'].max()} days")
print(f"  - Mean: {df_final['campaign_duration'].mean():.2f} days")
print(f"\nTarget dana statistics:")
print(f"  - Min: Rp {df_final['target_dana'].min():,.0f}")
print(f"  - Max: Rp {df_final['target_dana'].max():,.0f}")
print(f"  - Mean: Rp {df_final['target_dana'].mean():,.0f}")
print(f"\nProgress statistics:")
print(f"  - Min: {df_final['progress'].min():.6f}")
print(f"  - Max: {df_final['progress'].max():.6f}")
print(f"  - Mean: {df_final['progress'].mean():.6f}")

Final eval dataset structure:
  - Shape: (5, 10)
  - Columns: ['campaign', 'N', 'campaign_duration', 'target_dana', 'progress', 'p1', 'p2', 'p3', 'p4', 'p5']

Final dataset:
                   campaign    N  campaign_duration  target_dana  progress   p1   p2   p3   p4   p5
donors_20251221_202932.json   36                 24  40900000.00      0.01 0.86 0.08 0.06 0.00 0.00
donors_20251221_202957.json   50                 10 100000000.00      0.03 0.72 0.12 0.08 0.06 0.02
donors_20251221_203035.json  416                598 426000000.00      0.02 0.71 0.19 0.07 0.02 0.01
donors_20251221_203213.json 1675                740 500000000.00      0.08 0.74 0.04 0.19 0.01 0.01
donors_20251221_203249.json  253               4411 300000000.00      0.01 0.77 0.09 0.06 0.06 0.02

Eval Dataset Summary
Total campaigns: 5
Total donors: 2,430

Campaign duration statistics:
  - Min: 10 days
  - Max: 4411 days
  - Mean: 1156.60 days

Target dana statistics:
  - Min: Rp 40,900,000
  - Max: Rp 500,000,000
  -

## 6. Save Results

In [13]:
# Save to CSV
output_file = output_folder / 'data_preparation_eval_result.csv'
df_final.to_csv(output_file, index=False)

print(f"✓ Eval dataset saved successfully!")
print(f"  - File: {output_file}")
print(f"  - Shape: {df_final.shape}")
print(f"  - Size: {os.path.getsize(output_file) / 1024:.2f} KB")

print(f"\n{'='*60}")
print("✓ Data preparation for EVAL set completed!")
print(f"{'='*60}")

✓ Eval dataset saved successfully!
  - File: data_preparation_eval_result.csv
  - Shape: (5, 10)
  - Size: 0.76 KB

✓ Data preparation for EVAL set completed!
